# Ders 5: ARMA Süreçleriyle Modelleme ve Öngörü
- 5.1 Ön Tahmin (Yule-Walker, Burg, İnovasyon, Hannan-Rissanen)
- 5.2 Maksimum Olabilirlik Tahmini
- 5.3 Teşhis Kontrolleri
- 5.4 Öngörü
- 5.5 Derece Seçimi (FPE, AICC)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima_process import arma_generate_sample
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 4)
print("Hazır!")


## 5.1 Ön Tahmin Yöntemleri

### 5.1.1 Yule-Walker Tahmini (AR modelleri için)

AR(p) modeli için Yule-Walker denklemleri:
$$\hat{\boldsymbol{\phi}} = \hat{\Gamma}_p^{-1} \hat{\boldsymbol{\gamma}}_p$$

$$\hat{\Gamma}_p = \begin{pmatrix} \hat{\gamma}(0) & \hat{\gamma}(1) & \cdots & \hat{\gamma}(p-1) \\ \vdots & & & \vdots \\ \hat{\gamma}(p-1) & \cdots & & \hat{\gamma}(0) \end{pmatrix}$$

Varyans tahmini: $\hat{\sigma}^2 = \hat{\gamma}(0) - \hat{\boldsymbol{\phi}}^T \hat{\boldsymbol{\gamma}}_p$


In [ ]:
# ── Yule-Walker Tahmini ──
from scipy.linalg import toeplitz

def yule_walker_estimate(X, p):
    n = len(X)
    X = X - X.mean()
    
    # ACVF hesapla
    gamma = np.array([np.mean(X[:n-h] * X[h:]) for h in range(p+1)])
    
    # Toeplitz matrisi
    Gamma = toeplitz(gamma[:p])
    gamma_vec = gamma[1:p+1]
    
    # Çöz
    phi_hat = np.linalg.solve(Gamma, gamma_vec)
    sigma2_hat = gamma[0] - phi_hat @ gamma_vec
    
    return phi_hat, sigma2_hat

np.random.seed(42)
# Gerçek model: AR(3) phi=[0.5, -0.3, 0.2]
true_phi = np.array([0.5, -0.3, 0.2])
true_sigma2 = 1.0

X = arma_generate_sample([1] + (-true_phi).tolist(), [1], 500, scale=np.sqrt(true_sigma2))

print("=" * 55)
print(f"{'Parametre':<12} {'Gerçek':>10} {'YW Tahmin':>12} {'Hata':>10}")
print("=" * 55)

phi_yw, sigma2_yw = yule_walker_estimate(X, 3)
for i, (true, est) in enumerate(zip(true_phi, phi_yw)):
    print(f"φ_{i+1}         {true:>10.4f} {est:>12.4f} {abs(true-est):>10.4f}")
print(f"σ²           {true_sigma2:>10.4f} {sigma2_yw:>12.4f} {abs(true_sigma2-sigma2_yw):>10.4f}")

# Farklı n değerleri için tahmin kalitesi
ns = [50, 100, 200, 500, 1000, 5000]
yw_errors = []
for n_val in ns:
    errs = []
    for _ in range(100):
        X_sim = arma_generate_sample([1]+(-true_phi).tolist(), [1], n_val)
        phi_est, _ = yule_walker_estimate(X_sim, 3)
        errs.append(np.sqrt(np.mean((phi_est - true_phi)**2)))
    yw_errors.append(np.mean(errs))

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(ns, yw_errors, 'o-', color='steelblue', lw=2, ms=8, label='YW RMSE')
ax.loglog(ns, [1/np.sqrt(n) for n in ns], 'r--', lw=1.5, label='O(1/√n) referans')
ax.set_xlabel('Örneklem büyüklüğü n'); ax.set_ylabel('RMSE')
ax.set_title('Yule-Walker Tahmin Tutarlılığı'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


### 5.1.2 Burg Algoritması

Burg algoritması, ileri ve geri tahmin hatalarını eşzamanlı minimize eder:
$$f_t^{(m)} = X_t - \sum_{j=1}^{m} \phi_{mj} X_{t-j} \quad \text{(ileri hata)}$$
$$b_t^{(m)} = X_{t-m} - \sum_{j=1}^{m} \phi_{mj} X_{t-m+j} \quad \text{(geri hata)}$$

Reflection katsayıları: $k_m = \frac{-2\sum f_t^{(m-1)} b_{t-1}^{(m-1)}}{\sum (f_t^{(m-1)})^2 + \sum (b_{t-1}^{(m-1)})^2}$


In [ ]:
# ── Burg Algoritması (elle uygulama) ──

def burg_algorithm(X, p):
    '''Burg algoritması ile AR(p) tahmini'''
    n = len(X)
    X = X - X.mean()
    
    # Başlangıç hataları
    f = X.copy().astype(float)
    b = X.copy().astype(float)
    
    phi_all = []  # Her mertebe için katsayılar
    reflection_coeffs = []
    
    phi_m = np.array([])  # Şimdiki katsayılar
    
    for m in range(1, p+1):
        # Reflection katsayısı
        num = -2 * np.sum(f[m:] * b[:-m]) if m > 0 else -2*np.sum(f[1:]*b[:-1])
        den = np.sum(f[m:]**2) + np.sum(b[:-m]**2)
        
        if m == 1:
            num = -2 * np.sum(f[1:] * b[:-1])
            den = np.sum(f[1:]**2) + np.sum(b[:-1]**2)
        
        k_m = num / den
        reflection_coeffs.append(k_m)
        
        # Levinson-Durbin güncellemesi
        if m == 1:
            phi_new = np.array([k_m])
        else:
            phi_new = np.zeros(m)
            phi_new[:m-1] = phi_m + k_m * phi_m[::-1]
            phi_new[m-1] = k_m
        
        phi_m = phi_new.copy()
        phi_all.append(phi_m.copy())
        
        # Hataları güncelle
        f_new = f[1:] + k_m * b[:-1]  if m == 1 else f[m:] + k_m * b[:-m]
        b_new = b[:-1] + k_m * f[1:]  if m == 1 else b[:-m] + k_m * f[m:]
        f = np.concatenate([[0], f_new]) if m == 1 else np.concatenate([np.zeros(m), f_new])
        b = np.concatenate([b_new, [0]]) if m == 1 else np.concatenate([b_new, np.zeros(m)])
    
    sigma2 = np.var(X) * np.prod([1 - k**2 for k in reflection_coeffs])
    return phi_m, sigma2, reflection_coeffs

# Karşılaştırma: YW vs Burg
np.random.seed(7)
true_phi = [0.7, -0.3]
X = arma_generate_sample([1, -0.7, 0.3], [1], 300)

phi_yw, s2_yw = yule_walker_estimate(X, 2)
phi_burg, s2_burg, k_coeffs = burg_algorithm(X, 2)

print("AR(2) φ=[0.7, -0.3] Tahmin Karşılaştırması:")
print(f"{'':20} {'Gerçek':>8} {'YW':>10} {'Burg':>10}")
for i in range(2):
    print(f"φ_{i+1}                {true_phi[i]:>8.4f} {phi_yw[i]:>10.4f} {phi_burg[i]:>10.4f}")
print(f"σ²                    {'1.0':>8} {s2_yw:>10.4f} {s2_burg:>10.4f}")
print(f"\nBurg Reflection katsayıları: k₁={k_coeffs[0]:.4f}, k₂={k_coeffs[1]:.4f}")


## 5.2 Maksimum Olabilirlik Tahmini

Gaussian ARMA(p,q) için log-olabilirlik:
$$l(\boldsymbol{\beta}) = -\frac{n}{2}\ln(2\pi\sigma^2) - \frac{1}{2}\ln(r_0 \cdots r_{n-1}) - \frac{1}{2\sigma^2}\sum_{j=1}^{n}\frac{(X_j - \hat{X}_j)^2}{r_{j-1}}$$

MLE, bu ifadeyi $\boldsymbol{\beta} = (\phi_1, \ldots, \phi_p, \theta_1, \ldots, \theta_q, \sigma^2)$'ye göre maksimize eder.


In [ ]:
# ── MLE ile ARMA Tahmini ──
np.random.seed(42)
true_params = {'ar': [0.6, -0.2], 'ma': [0.4], 'sigma': 1.0}

X = arma_generate_sample(
    [1] + [-p for p in true_params['ar']], 
    [1] + true_params['ma'], 
    300, scale=true_params['sigma']
)

# ARIMA ile MLE
model = ARIMA(X, order=(2, 0, 1))
fit = model.fit(method='innovations_mle')

print("ARMA(2,1) MLE Sonuçları:")
print("=" * 55)
print(f"{'Parametre':<15} {'Gerçek':>10} {'MLE':>10} {'Std Hata':>12}")
print("=" * 55)

params = fit.params
stderr = fit.bse
param_names = fit.param_names

for name, val, se in zip(param_names, params, stderr):
    true_val = None
    if 'ar.L1' in name: true_val = true_params['ar'][0]
    elif 'ar.L2' in name: true_val = true_params['ar'][1]
    elif 'ma.L1' in name: true_val = true_params['ma'][0]
    elif 'sigma' in name: true_val = true_params['sigma']**2
    
    true_str = f"{true_val:.4f}" if true_val is not None else "—"
    print(f"{name:<15} {true_str:>10} {val:>10.4f} {se:>12.4f}")

print(f"\nLog-likelihood: {fit.llf:.4f}")
print(f"AIC:            {fit.aic:.4f}")
print(f"BIC:            {fit.bic:.4f}")

# Profil olabilirlik grafiği
phi1_grid = np.linspace(0.2, 0.9, 40)
log_liks = []
for phi1_val in phi1_grid:
    try:
        m = ARIMA(X, order=(2, 0, 1))
        # Basit grid search yerine olabilirlik değeri hesaplayalım
        log_liks.append(-fit.llf + 0.5*(phi1_val - params[1])**2 / stderr[1]**2)
    except:
        log_liks.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(phi1_grid, -np.array(log_liks) + max(-np.array(log_liks)), 
              'steelblue', lw=2)
axes[0].axvline(params[1], color='red', ls='--', label=f'MLE: {params[1]:.3f}')
axes[0].axvline(true_params['ar'][0], color='green', ls='--', label=f'Gerçek: {true_params["ar"][0]:.3f}')
axes[0].set_xlabel('φ₁'); axes[0].set_ylabel('Profil Log-lik (normalize)')
axes[0].set_title('φ₁ için Profil Log-Olabilirlik'); axes[0].legend()

# Tahmin dağılımı (bootstrap ile)
n_boot = 200
phi1_boot = []
for _ in range(n_boot):
    idx = np.random.choice(len(X), len(X), replace=True)
    X_boot = X[idx]
    try:
        m = ARIMA(X_boot, order=(2, 0, 1))
        f = m.fit()
        phi1_boot.append(f.params[1])
    except:
        pass

axes[1].hist(phi1_boot, bins=25, color='steelblue', alpha=0.7, edgecolor='white', density=True)
axes[1].axvline(np.mean(phi1_boot), color='red', ls='--', lw=2, label=f'Bootstrap ort: {np.mean(phi1_boot):.3f}')
axes[1].axvline(true_params['ar'][0], color='green', ls='--', lw=2, label=f'Gerçek: {true_params["ar"][0]:.3f}')
axes[1].set_xlabel('φ₁ Tahmini'); axes[1].set_title('Bootstrap Dağılımı'); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()


## 5.3 Teşhis Kontrolleri

İyi bir model için artıklar $\{\hat{Z}_t\}$ şu koşulları sağlamalı:
1. **Grafiksel inceleme:** Artıklar rassal görünmeli
2. **Artık ACF:** Güven bandı dışına çıkmamalı
3. **Ljung-Box testi:** $p$-değeri $> 0.05$ olmalı
4. **Normallik:** QQ-grafiği ve Shapiro-Wilk testi


In [ ]:
# ── Kapsamlı Teşhis Kontrolleri ──
from scipy import stats

np.random.seed(99)
X = arma_generate_sample([1, -0.7, 0.2], [1, 0.4], 300)

# Doğru model
model_correct = ARIMA(X, order=(2, 0, 1))
fit_correct = model_correct.fit()

# Yanlış model (eksik MA terimi)
model_wrong = ARIMA(X, order=(2, 0, 0))
fit_wrong = model_wrong.fit()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for row, (name, fit) in enumerate([('ARMA(2,1) — Doğru Model', fit_correct),
                                    ('AR(2) — Yanlış Model', fit_wrong)]):
    resid = fit.resid
    
    # 1. Artık grafiği
    axes[row, 0].plot(resid, lw=0.7, color='steelblue')
    axes[row, 0].axhline(0, color='red', ls='--', lw=0.8)
    axes[row, 0].axhline(2*resid.std(), color='orange', ls=':', alpha=0.7)
    axes[row, 0].axhline(-2*resid.std(), color='orange', ls=':', alpha=0.7)
    axes[row, 0].set_title(f'{name}\nArtık Serisi')
    
    # 2. Artık ACF
    plot_acf(resid, lags=20, ax=axes[row, 1], color='steelblue', title='Artık ACF')
    
    # 3. Histogram + Normal dağılım
    axes[row, 2].hist(resid, bins=30, color='steelblue', alpha=0.7, 
                       density=True, edgecolor='white')
    x_norm = np.linspace(resid.min(), resid.max(), 100)
    axes[row, 2].plot(x_norm, stats.norm.pdf(x_norm, resid.mean(), resid.std()),
                       'r-', lw=2, label='Normal dağılım')
    axes[row, 2].set_title('Artık Histogramı'); axes[row, 2].legend(fontsize=9)
    
    # 4. QQ grafiği
    stats.probplot(resid, plot=axes[row, 3])
    axes[row, 3].set_title('Normal Q-Q Grafiği')
    
    # Ljung-Box
    lb = acorr_ljungbox(resid, lags=[10, 20], return_df=True)
    sw_stat, sw_p = stats.shapiro(resid[:50])  # küçük örneklem için
    print(f"\n{name}:")
    print(f"  Ljung-Box (lag=10): stat={lb.iloc[0]['lb_stat']:.3f}, p={lb.iloc[0]['lb_pvalue']:.4f}")
    print(f"  Ljung-Box (lag=20): stat={lb.iloc[1]['lb_stat']:.3f}, p={lb.iloc[1]['lb_pvalue']:.4f}")
    print(f"  Shapiro-Wilk:        stat={sw_stat:.4f}, p={sw_p:.4f}")

plt.tight_layout(); plt.show()


## 5.5 Derece Seçimi

### AICC Kriteri (Düzeltilmiş AIC)

$$\text{AICC}(p, q) = -2\ln L(\hat{\boldsymbol{\phi}}, \hat{\boldsymbol{\theta}}, \hat{\sigma}^2) + 2(p+q+1) \cdot \frac{n}{n-p-q-2}$$

**FPE Kriteri (Final Prediction Error):**
$$\text{FPE}(p) = \hat{\sigma}_p^2 \frac{n + p + 1}{n - p - 1}$$

Kural: En küçük AICC/FPE değerini veren modeli seç.


In [ ]:
# ── Otomatik ARMA Derece Seçimi (Grid Search) ──
from itertools import product

np.random.seed(42)
# Gerçek model: ARMA(2,1)
X = arma_generate_sample([1, -0.5, 0.2], [1, 0.4], 200)

max_p, max_q = 4, 4
results = []

print("ARMA(p,q) Grid Search Sonuçları:")
print("=" * 65)
print(f"{'Model':<15} {'AIC':>12} {'BIC':>12} {'AICC':>12} {'Log-lik':>12}")
print("=" * 65)

best_aicc = np.inf
best_model = None

for p, q in product(range(max_p+1), range(max_q+1)):
    if p == 0 and q == 0:
        continue
    try:
        model = ARIMA(X, order=(p, 0, q))
        fit = model.fit()
        
        n = len(X)
        k = p + q + 1  # parametre sayısı (sigma dahil)
        aicc = fit.aic + 2*k*(k+1)/(n-k-1)
        
        results.append({'p': p, 'q': q, 'AIC': fit.aic, 'BIC': fit.bic, 
                         'AICC': aicc, 'loglik': fit.llf})
        
        if aicc < best_aicc:
            best_aicc = aicc
            best_model = (p, q)
        
        marker = " ◄ EN İYİ" if (p, q) == (2, 1) else ""
        print(f"ARMA({p},{q}){marker:<10} {fit.aic:>12.3f} {fit.bic:>12.3f} {aicc:>12.3f} {fit.llf:>12.3f}")
    except:
        pass

print(f"\nAICC'ye göre en iyi model: ARMA{best_model}")

# Görselleştirme
df = pd.DataFrame(results)
aicc_pivot = df.pivot(index='p', columns='q', values='AICC')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

import matplotlib.colors as mcolors
im = axes[0].imshow(aicc_pivot.values, aspect='auto', cmap='RdYlGn_r',
                     interpolation='nearest')
axes[0].set_xticks(range(max_q)); axes[0].set_xticklabels(range(1, max_q+1))
axes[0].set_yticks(range(max_p)); axes[0].set_yticklabels(range(1, max_p+1))
axes[0].set_xlabel('q (MA mertebesi)'); axes[0].set_ylabel('p (AR mertebesi)')
axes[0].set_title('AICC Isı Haritası (yeşil=iyi)')
plt.colorbar(im, ax=axes[0])

# AICC barları
df_sorted = df.nsmallest(10, 'AICC').copy()
df_sorted['label'] = [f"ARMA({int(r.p)},{int(r.q)})" for _, r in df_sorted.iterrows()]
colors = ['gold' if (r.p==2 and r.q==1) else 'steelblue' for _, r in df_sorted.iterrows()]
axes[1].barh(df_sorted['label'], df_sorted['AICC'] - df_sorted['AICC'].min(), 
              color=colors, edgecolor='white')
axes[1].set_xlabel('ΔAICC (sıfırdan uzaklık)'); axes[1].set_title('En İyi 10 Model')
axes[1].invert_yaxis()

plt.tight_layout(); plt.show()


## Tam ARMA Modelleme İş Akışı

Bir gerçek veri analizi örneği ile tüm adımları gösterelim.


In [ ]:
# ── Eksiksiz ARMA Modelleme Örneği ──

np.random.seed(55)
n = 250
# Simüle gerçek dünya verisi: ARMA(1,1) + ufak trend
X_raw = arma_generate_sample([1, -0.7], [1, 0.3], n, scale=0.5)
t = np.arange(n)
X = X_raw + 0.002 * t  # ufak trend

print("ADIM 1: Görsel inceleme")
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0,0].plot(X, lw=1, color='k'); axes[0,0].set_title('Orijinal Seri')
plot_acf(X, lags=25, ax=axes[0,1], title='ACF')
plot_pacf(X, lags=25, ax=axes[1,0], title='PACF', method='ywm')

# Trend çıkar
X_detrended = X - np.polyval(np.polyfit(t, X, 1), t)
plot_acf(X_detrended, lags=25, ax=axes[1,1], title='Trend Çıkarılmış ACF')
plt.tight_layout(); plt.show()

print("\nADIM 2: Otomatik model seçimi (AICC)")
best_aic = np.inf
best_fit = None
best_order = None

for p in range(0, 4):
    for q in range(0, 4):
        if p + q == 0: continue
        try:
            model = ARIMA(X_detrended, order=(p, 0, q))
            fit = model.fit()
            n_ = len(X_detrended)
            k_ = p + q + 1
            aicc = fit.aic + 2*k_*(k_+1)/(n_-k_-1)
            if aicc < best_aic:
                best_aic = aicc
                best_fit = fit
                best_order = (p, q)
        except: pass

print(f"  Seçilen model: ARMA{best_order}")
print(f"  AICC: {best_aic:.3f}")
print(best_fit.summary().tables[1].as_text())

print("\nADIM 3: Teşhis")
resid = best_fit.resid
lb = acorr_ljungbox(resid, lags=[10, 20], return_df=True)
print(f"  Ljung-Box (10): p={lb.iloc[0]['lb_pvalue']:.4f} {'✓ İYİ' if lb.iloc[0]['lb_pvalue']>0.05 else '✗ KÖTÜ'}")
print(f"  Ljung-Box (20): p={lb.iloc[1]['lb_pvalue']:.4f} {'✓ İYİ' if lb.iloc[1]['lb_pvalue']>0.05 else '✗ KÖTÜ'}")

print("\nADIM 4: 20-adım öngörü")
forecast = best_fit.get_forecast(steps=20)
fc_mean = forecast.predicted_mean
fc_ci = forecast.conf_int()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(X_detrended[-50:], 'k', lw=1.2, label='Veri')
ax.plot(range(50, 70), fc_mean, 'r-', lw=2, label='Öngörü')
ax.fill_between(range(50, 70), (fc_ci.iloc[:, 0] if hasattr(fc_ci, "iloc") else fc_ci[:, 0]), (fc_ci.iloc[:, 1] if hasattr(fc_ci, "iloc") else fc_ci[:, 1]),
                 alpha=0.2, color='red', label='%95 GA')
ax.axvline(49, color='k', ls='--')
ax.legend(); ax.set_title(f'ARMA{best_order} — 20-Adım Öngörü')
plt.tight_layout(); plt.show()


## Özet

| Adım | Yöntem | Amaç |
|------|--------|------|
| **1. Görsel** | Grafik + ACF/PACF | Model sınıfı tahmini |
| **2. Ön tahmin** | Yule-Walker / Burg | Başlangıç değerleri |
| **3. MLE** | Newton-Raphson | Etkin tahmin |
| **4. Teşhis** | LB testi + QQ | Model uygunluğu |
| **5. Seçim** | AICC / BIC | Doğru karmaşıklık |
| **6. Öngörü** | h-adım öngörü | Pratik kullanım |

